In [2]:
pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
pip install pandas

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ------------ --------------------------- 3.9/12.4 MB 46.1 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 63.7 MB/s  0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries and set path 

In [58]:
import polars as pl
from pathlib import Path
import fastexcel
import pandas as pd
import xlsxwriter

### Function Definition 

In [3]:
def read_excel(path: Path, sheet_pattern: str | None = None, had_header: bool = True) -> pl.DataFrame:
    """
    Read matching sheets from an Excel file into a single Polars DataFrame.
    
    - sheet_pattern: prefix to match sheet names (e.g. "Income" matches "Income - 1", "Income - 2").
                     None = read all sheets.
    - had_header: whether the sheet has a header row.
    """
    reader = fastexcel.read_excel(path)
    print(f"  Sheets in {path.name}: {reader.sheet_names}")

    # Filter sheets by pattern
    sheets = [s for s in reader.sheet_names
              if sheet_pattern is None or s.startswith(sheet_pattern)]
    print(f"  Matching sheets: {sheets}")

    dfs = []
    for name in sheets:
        df = pl.read_excel(path, sheet_name=name, has_header=had_header)
        df = df.with_columns(pl.lit(name).alias("_source_sheet"))
        dfs.append(df)

    if not dfs:
        return pl.DataFrame()

    combined = pl.concat(dfs, how="diagonal_relaxed") if len(dfs) > 1 else dfs[0]
    return combined.with_columns(pl.lit(path.name).alias("_source_file"))

In [4]:
def concat_files(files: list[Path], label: str, sheet_pattern: str | int | None = None, has_header: bool = True) -> pl.DataFrame:
    if not files:
        return pl.DataFrame()
    dfs = []
    for f in files:
        df = read_excel(f, sheet_pattern=sheet_pattern, had_header=has_header)
        if label == "income_all" and not has_header:
            header = df.row(2)
            df = df.slice(3)
            df.columns = header
            dfs.append(df)
        elif label == "balance_all" and not has_header:
            header =df.row(13)
            df = df.slice(14)
            df.columns = header
            dfs.append(df)
        print(f"  Loaded {f.name}: {df.shape}")
    combined = pl.concat(dfs, how="diagonal_relaxed")
    # before = combined.height
    # Drop duplicates (exclude _source_file so rows from overlapping files are caught)
    # data_cols = [c for c in combined.columns if c != "_source_file"]
    # combined = combined.unique(subset=data_cols, keep="first")
    # after = combined.height
    # dupes = before - after
    # print(f"=> {label}: {combined.shape} (removed {dupes} duplicate rows)\n")
    return combined

### Assign File locaiton

In [5]:
root = Path(r"c:\Users\TanJunJie\OneDrive - SRKK Group\Project\watson_entriesmatching\OneDrive_2026-03-09\Shopee Sample Reports (Testing)\scenario1")

# Find all .xlsx files recursively
xlsx_files = sorted(root.rglob("*.xlsx"))

### Bringing Income Report

In [6]:
# --- Load & concatenate by report type ---
income_files = [f for f in xlsx_files if f.name.startswith("Income.released")]
print("=== Income Released ===")
income_all = concat_files(income_files, "income_all", sheet_pattern="Income", has_header=False)



=== Income Released ===
  Sheets in Income.released.my.20251231_20251231.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Income - 1', 'Adjustment', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20251231_20251231.xlsx: (15021, 49)
  Sheets in Income.released.my.20260101_20260101.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260101_20260101.xlsx: (12599, 49)
  Sheets in Income.released.my.20260102_20260102.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260102_20260102.xlsx: (16012, 49)
  Sheets in Income.released.my.20260103_20260103.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']


In [7]:
# Filter to only "Order" view (exclude "Settlement" which has overlapping data but different granul
income_all.head()

Sequence No.,View By,Order ID,Shop Name,refund id,Product ID,Product Name,Order Creation Date,Payout Completed Date,Release Channel,Order Type,Hot Listing,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Amount Paid By Buyer,Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Shipping Fee Promotion by Seller,Shipping provider,Courier Name,Voucher Code From Seller,Lost Compensation,Cash refund to buyer amount,Pro-rated coin offset for return refund Items,Pro-rated Shopee voucher offset for returned items,Pro-rated Bank Payment Channel Promotion for return refund Items,Pro-rated Shopee Payment Channel Promotion for return refund Items,Income - 1,Income.released.my.20251231_20251231.xlsx,Income.released.my.20260101_20260101.xlsx,Income.released.my.20260102_20260102.xlsx,Income.released.my.20260103_20260103.xlsx,Income.released.my.20260104_20260104.xlsx,Income.released.my.20260105_20260105.xlsx,Income.released.my.20260106_20260106.xlsx,Income.released.my.20260107_20260107.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1""","""Order""","""251230QHF7YGBE""","""Watsons Malaysia""","""""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-5.19""","""27.93""","""-27.93""","""0""","""-4.9""","""-0.29""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""ndkshidshisbssayacantik""","""27.93""","""3.78""","""Cash on Delivery""","""""","""""","""0.00""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""0.00""","""-27.93""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""2""","""Sku""","""251230QHF7YGBE""","""Watsons Malaysia""","""-""","""28850540679""","""DOVE Body Scrub Pomegranate 28…","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-5.19""","""0""","""-27.93""","""0""","""-4.9""","""-0.29""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""-""","""""","""-""","""-""","""Cash on Delivery""","""""","""""","""-""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""-""","""-""","""-""","""-""","""-""","""-""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""3""","""Order""","""251230PMP6MTW6""","""Watsons Malaysia""","""220867667213152""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-2.65""","""14.9""","""-14.9""","""0""","""-2.5""","""-0.15""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""gq6pe2qreq""","""14.90""","""3.78""","""Online Banking""","""""","""""","""0.00""","""Self Collection Point""","""Self Collection Point (SPX Exp…","""""","""0.00""","""-14.90""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""4""","""Sku""","""251230PMP6MTW6""","""Watsons Malaysia""","""-""","""16492683485""","""Orita Charcoal Dehumidifier (3…","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-2.65""","""0""","""-14.9""","""0""","""-2

In [8]:
income_all.columns

['Sequence No.',
 'View By',
 'Order ID',
 'Shop Name',
 'refund id',
 'Product ID',
 'Product Name',
 'Order Creation Date',
 'Payout Completed Date',
 'Release Channel',
 'Order Type',
 'Hot Listing',
 'Total Released Amount (RM)',
 'Product Price',
 'Refund Amount',
 'Shipping Fee Paid by Buyer (excl. SST)',
 'Shipping Fee Charged by Logistic Provider',
 'Seller Paid Shipping Fee SST',
 'Shipping Rebate From Shopee',
 'Reverse Shipping Fee',
 'Reverse Shipping Fee SST',
 'Saver Programme Shipping Fee Savings',
 'Return to Seller Fee',
 'Rebate Provided by Shopee',
 'Voucher Sponsored by Seller',
 'Coin Cashback Sponsored by Seller',
 'Commission Fee (incl. SST)',
 'Service Fee (Incl. SST)',
 'Transaction Fee (Incl. SST)',
 'AMS Commission Fee',
 'Saver Programme Fee (Incl. SST)',
 'Username (Buyer)',
 'Amount Paid By Buyer',
 'Transaction Fee Rate (%)',
 'Buyer Payment Method',
 'Buyer Payment Method Details_1(if applicable)',
 'Payment Details / Installment Plan',
 'Shipping Fee Pr

In [27]:
#Select only relevant columns for matching (exclude "View By" which is used for filtering but not needed in matching)
income_all_normalized = (
    income_all.
    filter(pl.col("View By")== "Order")
    .select(
    ['Sequence No.',
    'Order ID',
    'Order Creation Date',
    'Payout Completed Date',

    'Total Released Amount (RM)',
    'Product Price',
    'Refund Amount',
    'Shipping Fee Paid by Buyer (excl. SST)',
    'Shipping Fee Charged by Logistic Provider',
    'Seller Paid Shipping Fee SST',
    'Shipping Rebate From Shopee',
    'Reverse Shipping Fee',
    'Reverse Shipping Fee SST',
    'Saver Programme Shipping Fee Savings',
    'Return to Seller Fee',
    'Rebate Provided by Shopee',
    'Voucher Sponsored by Seller',
    'Coin Cashback Sponsored by Seller',
    'Commission Fee (incl. SST)',
    'Service Fee (Incl. SST)',
    'Transaction Fee (Incl. SST)',
    'AMS Commission Fee',
    'Saver Programme Fee (Incl. SST)',

    'Username (Buyer)',
    'Transaction Fee Rate (%)',
    'Buyer Payment Method',
    'Buyer Payment Method Details_1(if applicable)',
    'Payment Details / Installment Plan',

    'Voucher Code From Seller',
    'Lost Compensation',
    ])
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col('Total Released Amount (RM)').cast(pl.Float64, strict=False),
        pl.col('Order Creation Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Payout Completed Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Product Price').cast(pl.Float64, strict=False),
        pl.col('Refund Amount').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Paid by Buyer (excl. SST)').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Charged by Logistic Provider').cast(pl.Float64, strict=False),
        pl.col('Seller Paid Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Shipping Rebate From Shopee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Shipping Fee Savings').cast(pl.Float64, strict=False),
        pl.col('Return to Seller Fee').cast(pl.Float64, strict=False),
        pl.col('Rebate Provided by Shopee').cast(pl.Float64, strict=False),
        pl.col('Voucher Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Coin Cashback Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Commission Fee (incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Service Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('AMS Commission Fee').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee Rate (%)').cast(pl.Float64, strict=False),
        pl.col('Lost Compensation').cast(pl.Float64, strict=False),
    ])
    .drop_nulls(['Order ID'])
    )
income_all_normalized.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64
"""1""","""251230QHF7YGBE""",2025-12-30,2025-12-31,-5.19,27.93,-27.93,0.0,-4.9,-0.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ndkshidshisbssayacantik""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0
"""3""","""251230PMP6MTW6""",2025-12-30,2025-12-31,-2.65,14.9,-14.9,0.0,-2.5,-0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gq6pe2qreq""",3.78,"""Online Banking""","""""","""""","""""",0.0
"""5""","""251230PB5YG8HT""",2025-12-30,2025-12-31,37.32,44.1,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.64,0.0,-2.14,0.0,0.0,"""fiqamokhtar""",4.86,"""SPayLater""","""""","""12x""","""""",0.0
"""7""","""251230PB3GD63E""",2025-12-30,2025-12-31,34.51,41.04,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.54,0.0,-1.99,0.0,0.0,"""nurnabilahabukasim""",4.86,"""SPayLater""","""""","""3x""","""""",0.0
"""10""","""251230PAY9XQWY""",2025-12-30,2025-12-31,26.79,31.5,0.0,4.9,-4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.32,0.0,-1.39,0.0,0.0,"""amyrahbrown""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0


### Bringing Balance Transaction Report

In [10]:
balance_files = [f for f in xlsx_files if f.name.startswith("my_balance_transaction")]
print("=== Balance Transaction ===")
balance_all = concat_files(balance_files, "balance_all", sheet_pattern="Transaction", has_header=False)

=== Balance Transaction ===
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: (5000, 10)
  Sheets

In [11]:
balance_all.columns

['Date',
 'Transaction Type',
 'Description',
 'Order ID',
 'Money Direction',
 'Amount',
 'Status',
 'Balance After Transactions',
 'Transaction Report',
 'my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx']

In [12]:

balance_all_normalized = (
    balance_all.
        select(['Date',
        'Transaction Type',
        'Description',
        'Order ID'])
        .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col("Date").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
    ])
        
)
balance_all_normalized.head(10)

Date,Transaction Type,Description,Order ID
datetime[μs],str,str,str
2026-01-07 23:59:53,"""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP"""
2026-01-07 23:59:46,"""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV"""
2026-01-07 23:59:32,"""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG"""
2026-01-07 23:59:28,"""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6"""
2026-01-07 23:59:26,"""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU"""
2026-01-07 23:59:06,"""Order Income""","""Income from Order #26010575HF5…","""26010575HF5Q6D"""
2026-01-07 23:59:02,"""Order Income""","""Income from Order #26010347C47…","""26010347C470NX"""
2026-01-07 23:58:33,"""Order Income""","""Income from Order #260101UAHS8…","""260101UAHS8FRR"""
2026-01-07 23:58:22,"""Order Income""","""Income from Order #2601069VSE7…","""2601069VSE7UX0"""


In [85]:
balance_all.select(pl.col("Status").unique())

Status
str
"""Transaction Completed"""


In [88]:
balance_all.filter(pl.col("Order ID")== "251227F28P4MVX")

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str


In [14]:
balance_all_withdrawal = balance_all_normalized.filter(pl.col("Transaction Type").is_in(["Withdrawals"]))

In [89]:
path = "../Shopee Payment Master List 180126.xlsx"
recon = pl.read_excel(path,sheet_name="Recon", has_header=True)
recon.head()

Could not determine dtype for column 19, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 25, falling back to string
Could not determine dtype for column 26, falling back to string
Could not determine dtype for column 27, falling back to string
Could not determine dtype for column 28, falling back to string
Could not determine dtype for column 29, falling back to string
Could not determine dtype for column 30, falling back to string
Could not determine dtype for column 31, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 33, falling back to string
Could not determine dtype for column 34, falling back to string
Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 38,

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str
"""9523929588""","""250705B8MM4UH4""","""2025-07-06""",37.18,2025-07-01,2025-07-23,2025-07-01,23.45,2.87,1.61,0.0,0.0,0,0.0,0.0,0.0,9.25,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9.25,null
"""9523802829""","""250612C1TJ9SHE""","""2025-07-07""",-3.0,2025-07-01,2025-06-25,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-3.0,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-3.0,null
"""9523757815""","""250604N6AD379X""","""2025-07-07""",-9.73,2025-07-01,2025-06-18,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-9.73,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-9.73,null
"""9523971310""","""250711TY92VWPU""","""2025-07-13""",153.78,2025-07-01,2025-07-23,2025-07-01,128.05,15.48,7.36,0.0,0.0,0,11.0,0.0,8.8818e-16,2.89,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.89,null
"""9523975421""","""2507120F7GFHVV""","""2025-07-14""",278.03,2025-07-01,2025-07-30,2025-08-01,237.96,28.53,10.59,0.0,0.0,0,16.0,0.0,0.0,0.95,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.95,null


In [91]:
recon = recon.with_columns(pl.col("MarketPlaceOrderNum").cast(pl.Utf8).str.strip_chars().alias("Order ID"))
recon.head()

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.,Order ID
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""9523929588""","""250705B8MM4UH4""","""2025-07-06""",37.18,2025-07-01,2025-07-23,2025-07-01,23.45,2.87,1.61,0.0,0.0,0,0.0,0.0,0.0,9.25,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9.25,null,"""250705B8MM4UH4"""
"""9523802829""","""250612C1TJ9SHE""","""2025-07-07""",-3.0,2025-07-01,2025-06-25,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-3.0,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-3.0,null,"""250612C1TJ9SHE"""
"""9523757815""","""250604N6AD379X""","""2025-07-07""",-9.73,2025-07-01,2025-06-18,2025-06-01,0.0,0.0,0.0,0.0,0.0,0,0.0,null,0.0,-9.73,"""Check with Marcus""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-9.73,null,"""250604N6AD379X"""
"""9523971310""","""250711TY92VWPU""","""2025-07-13""",153.78,2025-07-01,2025-07-23,2025-07-01,128.05,15.48,7.36,0.0,0.0,0,11.0,0.0,8.8818e-16,2.89,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.89,null,"""250711TY92VWPU"""
"""9523975421""","""2507120F7GFHVV""","""2025-07-14""",278.03,2025-07-01,2025-07-30,2025-08-01,237.96,28.53,10.59,0.0,0.0,0,16.0,0.0,0.0,0.95,"""Unknown""","""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.95,null,"""2507120F7GFHVV"""


In [104]:
recon.filter(pl.col("Order ID") == "2601045R7NTP28")

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.,Order ID
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""9524951072""","""2601045R7NTP28""","""2026-01-05 00:00:00""",107.65,2026-01-01,2026-01-07,2026-01-01,91.35,11.07,5.23,0.0,0.0,0,9.0,0.0,0.0,1.0658e-14,null,"""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0658e-14,null,"""2601045R7NTP28"""


In [105]:
recon.filter(pl.col("Order ID") == "260101UF58WU38")

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.,Order ID
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""9524930857""","""260101UF58WU38""","""2026-01-05 00:00:00""",2.73,2026-01-01,2026-01-07,2026-01-01,2.33,0.3,0.1,0.0,0.0,0,0.0,0.0,0.0,-8.3267e-17,null,"""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-8.3267e-17,null,"""260101UF58WU38"""


In [116]:
recon.filter(
	(pl.col("Actual Shipping Fee") == 1.06) &
	(pl.col("Pymt Date") == pl.date(2026, 1, 14))
)

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.,Order ID
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""9524982329""","""260110P1CNKDKW""","""2026-01-12 00:00:00""",141.96,2026-01-01,2026-01-14,2026-01-01,124.06,11.11,5.73,0.0,0.0,0,15.0,0.0,1.06,5.7732e-15,null,"""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5.7732e-15,null,"""260110P1CNKDKW"""
"""9524982329""","""260110P1CNKDKW""","""2026-01-12 00:00:00""",141.96,2026-01-01,2026-01-14,2026-01-01,124.06,11.11,5.73,0.0,0.0,0,15.0,0.0,1.06,5.7732e-15,null,"""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5.7732e-15,null,"""260110P1CNKDKW"""


In [117]:
income_all_normalized.filter(pl.col("Order ID") == "260110P1CNKDKW").head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64


In [103]:
recon.columns

['OrderNum',
 'MarketPlaceOrderNum',
 'WorkDate',
 'SalesCenterAmount',
 'Sales Month',
 'Pymt Date',
 'Pymt Mth',
 'Pymt Amt',
 'Commission Fee',
 'Transaction Fee',
 'Service Fee',
 'AMS Commission Fee',
 'Return QC Fee',
 'Voucher/(Disc rebate)',
 'Refund',
 'Actual Shipping Fee',
 'Outstanding',
 'Remarks',
 'Order Status',
 'Shipping fee paid by Shopee',
 'Lost Compensation',
 'Rebate Provided by Shopee',
 'Checking',
 'Refund Adj Mth 1',
 'Refund Adj 1',
 'Refund Adj Mth 2',
 'Refund Adj 2',
 'Shrinkage Adj Mth',
 'Shrinkage Adj',
 'Deduction of Affiliate Marketing Solution commission fee Adj Mth',
 'Deduction of Affiliate Marketing Solution commission fee Adj Mth 1',
 'Voucher Adj Mth',
 'Voucher Adj',
 'Shipping Fee Adj Mth 1',
 'Shipping Fee Adj 1',
 'Shipping Fee Adj Mth 2',
 'Shipping Fee Adj 2',
 'Oustanding Balance',
 '.',
 'Order ID']

In [95]:
checkreport = balance_all_withdrawal.join(recon, on="Order ID", how="inner")
checkreport.head(100)

Date,Transaction Type,Description,Order ID,OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.
datetime[μs],str,str,str,str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str


### Bringing Sales Report

In [15]:
sales_files = sorted(root.parent.glob("SalesReport*.xlsx"))
print("=== Sales Reports ===")
print([f.name for f in sales_files])

sales_dfs = []
for f in sales_files:
    df = pl.read_excel(f, sheet_name="SalesReport", has_header=True)
    df = df.with_columns(pl.lit(f.name).alias("_source_file"))
    sales_dfs.append(df)

sales_report = pl.concat(sales_dfs, how="diagonal_relaxed") if sales_dfs else pl.DataFrame()
print("sales_report shape:", sales_report.shape)

sales_report.head()

=== Sales Reports ===
['SalesReport Wk2.xlsx', 'SalesReport Wk3.xlsx']


Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string


sales_report shape: (62427, 10)


OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file
str,str,i64,i64,date,f64,str,str,str,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""


In [16]:
sales_report.columns

['OrderNum',
 'MarketPlaceOrderNum',
 'SalesGlobalTxnID',
 'eStoreCode',
 'SalesWorkDate',
 'TotalAmount',
 'PayId',
 'TenderType',
 'TlogCount',
 '_source_file']

In [79]:
# Normalize sales report
sales_report_normalized = (
    sales_report
    .with_columns([
        # `SalesWorkDate` may already be a Date/Datetime (from Excel reader),
        # or it may be a string like "31-12-2025"; handle both safely.
        pl.coalesce([
            pl.col("SalesWorkDate").cast(pl.Date, strict=False),
            pl.col("SalesWorkDate")
              .cast(pl.Utf8, strict=False)
              .str.strip_chars()
              .str.strptime(pl.Date, "%d-%m-%Y", strict=False),
        ]).alias("SalesWorkDate"),
        pl.col("MarketPlaceOrderNum").cast(pl.Utf8).str.strip_chars().alias("Order ID"),
    ])
)
sales_report_normalized.head()

OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file,Order ID
str,str,i64,i64,date,f64,str,str,str,str,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""2601044PUM2781"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""26010450MFPQE1"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""2601045R7NTP28"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""260101TUY6HNTV"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""260101UF58WU38"""


In [80]:
sales_report_report = sales_report_normalized.select(pl.col("Order ID", "SalesWorkDate"))
sales_report_report.head()

Order ID,SalesWorkDate
str,date
"""2601044PUM2781""",2026-01-05
"""26010450MFPQE1""",2026-01-05
"""2601045R7NTP28""",2026-01-05
"""260101TUY6HNTV""",2026-01-05
"""260101UF58WU38""",2026-01-05


In [96]:
# Get Order IDs from each dataframe
recon_ids = recon.select("Order ID")
sales_ids = sales_report_normalized.select("Order ID")
balance_ids = balance_all_normalized.select("Order ID")

# Find common Order IDs across all three
common_ids = (
    recon_ids
    .join(sales_ids, on="Order ID", how="inner")
    .join(balance_ids, on="Order ID", how="inner")
    .unique("Order ID")
)

print(f"Order IDs in all three sources: {common_ids.height}")
common_ids.head(20)

Order IDs in all three sources: 7862


Order ID
str
"""2601046NSCRFCU"""
"""26010349F27NWJ"""
"""2601045TNC0XS3"""
"""2601069PFCA9QB"""
"""2601020REFUY1G"""
…
"""2601045BG68B71"""
"""2601058A3N83H2"""
"""2601033R48K1XC"""


In [97]:
balance_all.filter(pl.col("Order ID") == "2601069PFCA9QB").head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2026-01-07 18:21:16""","""Order Income""","""Income from Order #2601069PFCA…","""2601069PFCA9QB""","""Money In""","""16.8""","""Transaction Completed""","""117684.9""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null


In [98]:
recon.filter(pl.col("Order ID") == "2601069PFCA9QB").head()

OrderNum,MarketPlaceOrderNum,WorkDate,SalesCenterAmount,Sales Month,Pymt Date,Pymt Mth,Pymt Amt,Commission Fee,Transaction Fee,Service Fee,AMS Commission Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding,Remarks,Order Status,Shipping fee paid by Shopee,Lost Compensation,Rebate Provided by Shopee,Checking,Refund Adj Mth 1,Refund Adj 1,Refund Adj Mth 2,Refund Adj 2,Shrinkage Adj Mth,Shrinkage Adj,Deduction of Affiliate Marketing Solution commission fee Adj Mth,Deduction of Affiliate Marketing Solution commission fee Adj Mth 1,Voucher Adj Mth,Voucher Adj,Shipping Fee Adj Mth 1,Shipping Fee Adj 1,Shipping Fee Adj Mth 2,Shipping Fee Adj 2,Oustanding Balance,.,Order ID
str,str,str,f64,date,date,date,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,str,str,str,str,f64,str,date,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
"""9524962601""","""2601069PFCA9QB""","""2026-01-07 00:00:00""",19.6,2026-01-01,2026-01-14,2026-01-01,16.8,2.06,0.74,0.0,0.0,0,0.0,0.0,0.0,6.6613e-16,null,"""""",null,"""""",0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6.6613e-16,null,"""2601069PFCA9QB"""


In [100]:
sales_report_normalized.filter(pl.col("Order ID") == "2601069PFCA9QB").head()

OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file,Order ID
str,str,i64,i64,date,f64,str,str,str,str,str
"""9524962601""","""2601069PFCA9QB""",234084936,895,2026-01-07,19.6,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""","""2601069PFCA9QB"""


In [101]:
income_all_normalized.filter(pl.col("Order ID") == "2601069PFCA9QB").head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""641""","""2601069PFCA9QB""",2026-01-06,2026-01-07,16.8,19.6,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.06,0.0,-0.74,0.0,0.0,"""nzpedna8ug""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0


In [102]:
income_all_normalized.select(pl.col("Transaction Fee (Incl. SST)")).head()

Transaction Fee (Incl. SST)
f64
0.0
0.0
-2.14
-1.99
-1.39


### Generate Report 

In [28]:
income_all_normalized.columns


['Sequence No.',
 'Order ID',
 'Order Creation Date',
 'Payout Completed Date',
 'Total Released Amount (RM)',
 'Product Price',
 'Refund Amount',
 'Shipping Fee Paid by Buyer (excl. SST)',
 'Shipping Fee Charged by Logistic Provider',
 'Seller Paid Shipping Fee SST',
 'Shipping Rebate From Shopee',
 'Reverse Shipping Fee',
 'Reverse Shipping Fee SST',
 'Saver Programme Shipping Fee Savings',
 'Return to Seller Fee',
 'Rebate Provided by Shopee',
 'Voucher Sponsored by Seller',
 'Coin Cashback Sponsored by Seller',
 'Commission Fee (incl. SST)',
 'Service Fee (Incl. SST)',
 'Transaction Fee (Incl. SST)',
 'AMS Commission Fee',
 'Saver Programme Fee (Incl. SST)',
 'Username (Buyer)',
 'Transaction Fee Rate (%)',
 'Buyer Payment Method',
 'Buyer Payment Method Details_1(if applicable)',
 'Payment Details / Installment Plan',
 'Voucher Code From Seller',
 'Lost Compensation']

In [29]:
# Add "Net Voucher" column as sum of all voucher-related columns (treating nulls as 0)
income_all_normalized = income_all_normalized.with_columns(
    pl.sum_horizontal([
        pl.col('Rebate Provided by Shopee'),
        pl.col('Voucher Sponsored by Seller'),
        pl.col('Coin Cashback Sponsored by Seller')
    ]).fill_null(0).alias("Net Voucher")
)

In [30]:
income_all_normalized = income_all_normalized.with_columns(
    pl.sum_horizontal([
        pl.col('Shipping Fee Paid by Buyer (excl. SST)'),
        pl.col('Shipping Fee Charged by Logistic Provider'),
        pl.col('Seller Paid Shipping Fee SST'),
        pl.col('Shipping Rebate From Shopee'),
        pl.col('Reverse Shipping Fee'),
        pl.col('Reverse Shipping Fee SST')
    ]).fill_null(0).alias("Net Shipping Fees")
)
income_all_normalized.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""1""","""251230QHF7YGBE""",2025-12-30,2025-12-31,-5.19,27.93,-27.93,0.0,-4.9,-0.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ndkshidshisbssayacantik""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,-5.19
"""3""","""251230PMP6MTW6""",2025-12-30,2025-12-31,-2.65,14.9,-14.9,0.0,-2.5,-0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gq6pe2qreq""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,-2.65
"""5""","""251230PB5YG8HT""",2025-12-30,2025-12-31,37.32,44.1,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.64,0.0,-2.14,0.0,0.0,"""fiqamokhtar""",4.86,"""SPayLater""","""""","""12x""","""""",0.0,0.0,0.0
"""7""","""251230PB3GD63E""",2025-12-30,2025-12-31,34.51,41.04,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.54,0.0,-1.99,0.0,0.0,"""nurnabilahabukasim""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0
"""10""","""251230PAY9XQWY""",2025-12-30,2025-12-31,26.79,31.5,0.0,4.9,-4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.32,0.0,-1.39,0.0,0.0,"""amyrahbrown""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0,0.0,0.0


In [31]:
balance_all_report = balance_all_normalized.select([
    'Date', 'Order ID'])
#Change the Date time to Date only for easier matching with income report (which only has date, no time)
balance_all_report = balance_all_report.with_columns(
    pl.col("Date").dt.date().alias("Payment Date")
)
balance_all_report = balance_all_report.with_columns(
    pl.col("Payment Date").dt.strftime("%d-%b-%Y").str.to_uppercase().alias("Payment Date")
)
balance_all_report = balance_all_report.with_columns(
    pl.col("Order ID").str.strip_chars(),
    pl.col("Payment Date")
)
balance_all_report.head()

Date,Order ID,Payment Date
datetime[μs],str,str
2026-01-07 23:59:53,"""251227FMQA3XGP""","""07-JAN-2026"""
2026-01-07 23:59:46,"""260102VN9VMVPV""","""07-JAN-2026"""
2026-01-07 23:59:32,"""2601045P6TK6PG""","""07-JAN-2026"""
2026-01-07 23:59:28,"""2601045Y4YTEG6""","""07-JAN-2026"""
2026-01-07 23:59:26,"""2601045E6UGTNU""","""07-JAN-2026"""


In [81]:
report = income_all_normalized.join(
    balance_all_report,
    on="Order ID",
    how="inner",
)
report.head(10)

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026"""
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026"""
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026"""
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026"""
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026"""
"""4176""","""26010575HF5Q6D""",2026-01-05,2026-01-07,14.92,17.52,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.75,0.0,-0.85,0.0,0.0,"""nurainsyazliyana""",4.86,"""SPayLater""","""""","""1x""","""""",0.0,0.0,0.0,2026-01-07 23:59:06,"""07-JAN-2026"""
"""7612""","""26010347C470NX""",2026-01-03,2026-01-07,16.71,19.5,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.05,0.0,-0.74,0.0,0.0,"""farizaalissya""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:02,"""07-JAN-2026"""
"""493""","""260101UAHS8FRR""",2026-01-01,2026-01-07,6.76,7.89,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.83,0.0,-0.3,0.0,0.0,"""shazulkiflii""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:58:33,"""07-JAN-2026"""
"""557""","""2601069VSE7UX0""",2026-01-06,2026-01-07,33.34,38.96,0.0,1.3,-6.3,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.1,0.0,-1.52,0.0,0.0,"""amirahizzati01""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:58:22,"""07-JAN-2026"""


In [82]:
report = report.with_columns(
     pl.col("Date").dt.strftime("%b-%Y").str.to_uppercase().alias("Payment Mth")
)
report.head()


Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026""","""JAN-2026"""
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026""","""JAN-2026"""
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026""","""JAN-2026"""
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026""","""JAN-2026"""
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026""","""JAN-2026"""


In [83]:
report.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str
"""1813""","""251227FMQA3XGP""",2025-12-27,2026-01-07,13.42,15.0,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.01,0.0,-0.57,0.0,0.0,"""tc016.5398""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:53,"""07-JAN-2026""","""JAN-2026"""
"""8862""","""260102VN9VMVPV""",2026-01-02,2026-01-07,8.31,10.14,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.12,-0.33,-0.38,0.0,0.0,"""limshinyee""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:46,"""07-JAN-2026""","""JAN-2026"""
"""6896""","""2601045P6TK6PG""",2026-01-04,2026-01-07,21.25,24.8,0.0,0.0,-3.9,0.0,3.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.61,0.0,-0.94,0.0,0.0,"""kemekthepuake""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 23:59:32,"""07-JAN-2026""","""JAN-2026"""
"""6428""","""2601045Y4YTEG6""",2026-01-04,2026-01-07,13.19,15.77,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.77,-0.18,0.0,"""liantan312""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:28,"""07-JAN-2026""","""JAN-2026"""
"""7123""","""2601045E6UGTNU""",2026-01-04,2026-01-07,23.08,27.45,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.04,0.0,-1.33,0.0,0.0,"""ainfrdz""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0,2026-01-07 23:59:26,"""07-JAN-2026""","""JAN-2026"""


In [84]:
PaymentDetail= report.join(
    sales_report_report,   
    on="Order ID",
    how="inner"
)
PaymentDetail.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees,Date,Payment Date,Payment Mth,SalesWorkDate
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64,datetime[μs],str,str,date
"""118""","""2601045R7NTP28""",2026-01-04,2026-01-05,91.35,116.65,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,-9.0,0.0,-11.07,0.0,-5.23,0.0,0.0,"""minnouri""",4.86,"""SPayLater""","""""","""6x""","""WATS0104A""",0.0,-9.0,0.0,2026-01-05 14:37:37,"""05-JAN-2026""","""JAN-2026""",2026-01-05
"""9263""","""260101UF58WU38""",2026-01-01,2026-01-05,2.33,2.73,0.0,0.0,-2.5,0.0,2.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.3,0.0,-0.1,0.0,0.0,"""sahsahhira750""",3.78,"""Payment waived""","""""","""""","""""",0.0,0.0,0.0,2026-01-05 14:52:46,"""05-JAN-2026""","""JAN-2026""",2026-01-05
"""8858""","""260102VNEQ4SRV""",2026-01-02,2026-01-07,13.18,15.48,0.0,2.0,-10.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.63,0.0,-0.67,0.0,0.0,"""xiivivilyn078""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0,0.0,0.0,2026-01-07 00:35:23,"""07-JAN-2026""","""JAN-2026""",2026-01-05
"""3774""","""2601044RABCD24""",2026-01-04,2026-01-06,83.28,106.8,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,-9.0,0.0,-9.77,0.0,-4.75,0.0,0.0,"""sitiaznol""",4.86,"""SPayLater""","""""","""1x""","""WATS0104A""",0.0,-9.0,0.0,2026-01-06 08:21:59,"""06-JAN-2026""","""JAN-2026""",2026-01-05
"""5702""","""2601021CAUPAJK""",2026-01-02,2026-01-05,27.79,32.43,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.41,0.0,-1.23,0.0,0.0,"""ucxp46""",3.78,"""ShopeePay Balance""","""""","""""","""""",0.0,0.0,0.0,2026-01-05 15:47:42,"""05-JAN-2026""","""JAN-2026""",2026-01-05


In [77]:
sales_report_report.head()

Order ID,SalesWorkDate
str,date
"""9524949786""",2026-01-05
"""9524950020""",2026-01-05
"""9524951072""",2026-01-05
"""9524928466""",2026-01-05
"""9524930857""",2026-01-05


In [63]:
income_not_balance = income_all_normalized.join(
    balance_all_normalized,
    on="Order ID",
    how="anti"
)
income_not_balance.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation,Net Voucher,Net Shipping Fees
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64,f64,f64
"""8832""","""2512222NWK4XVR""",2025-12-22,2025-12-31,0.0,66.0,-66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""aaadrw""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""9076""","""25122108AA8SM7""",2025-12-21,2025-12-31,0.0,39.8,-39.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""rajaindah5597""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""6067""","""251228GK5T85K4""",2025-12-28,2026-01-01,0.0,72.5,-72.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""_azto1n21w""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0
"""7582""","""251225AV1ET5T5""",2025-12-25,2026-01-01,0.0,22.66,-22.66,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""awimzjr""",4.86,"""SPayLater""","""""","""3x""","""""",0.0,0.0,0.0
"""8842""","""2512221YJBXS68""",2025-12-22,2026-01-01,0.0,8.58,-8.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""nuradilanazira""",3.78,"""Online Banking""","""""","""""","""""",0.0,0.0,0.0


In [64]:
balance_not_income = balance_all_normalized.join(
    income_all_normalized, 
    on="Order ID",
    how="anti"
)
balance_not_income.head()

Date,Transaction Type,Description,Order ID
datetime[μs],str,str,str
2026-01-07 18:18:01,"""Adjustment""","""Lost Compensation for parcel #…","""251227FHQ8CDPK"""
2026-01-07 17:18:27,"""Adjustment""","""Wallet Adjustment due to Buyer…","""251026534124X4"""
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-"""
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-"""
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-"""


### Export Result Out to Excel File

In [66]:
workbook = xlsxwriter.Workbook("output.xlsx")

# Function to write polars dataframe to worksheet
def write_polars_to_sheet(workbook, sheet_name, df):
    worksheet = workbook.add_worksheet(sheet_name)

    # Write header
    for col_idx, column in enumerate(df.columns):
        worksheet.write(0, col_idx, column)

    # Write data
    for row_idx, row in enumerate(df.iter_rows()):
        for col_idx, value in enumerate(row):
            worksheet.write(row_idx + 1, col_idx, value)

# Write each dataframe
write_polars_to_sheet(workbook, "report", report)
write_polars_to_sheet(workbook, "Income not in Balance", income_not_balance)
write_polars_to_sheet(workbook, "Balance not in Income", balance_not_income)

workbook.close()